In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!find '/content/drive' -name 'compliancemetrics.csv' -type f 2>/dev/null | head -5

Mounted at /content/drive


In [3]:
!find '/content/drive' -name 'compliancemetrics*' -type f 2>/dev/null | head -5
!find '/content/drive' -name 'trialsclean*' -type f 2>/dev/null | head -5

In [4]:
!find '/content/drive' -name 'compliancemetrics*' -type f 2>/dev/null | head -5

/content/drive/.shortcut-targets-by-id/1RHn-liAsgGH31DsJuv0hJrog3w2if1zt/Data/step 1 - clinical trial w spark/compliancemetrics.csv


In [5]:
!find '/content/drive' -name 'trialsclean*' -type f 2>/dev/null | head -5

/content/drive/.shortcut-targets-by-id/1RHn-liAsgGH31DsJuv0hJrog3w2if1zt/Data/step 1 - clinical trial w spark/trialsclean.csv


In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
import pandas as pd
import numpy as np
import glob
import os
import zipfile
import requests
import time
import re

BASE_SPARK = '/content/drive/.shortcut-targets-by-id/1RHn-liAsgGH31DsJuv0hJrog3w2if1zt/Data/step 1 - clinical trial w spark/'
BASE_OLD = '/content/drive/.shortcut-targets-by-id/1_XE0-4_DTuI64Jsm6Ngp3XmHmjt6H8QI/Clinical trial '

# Load new Spark compliance metrics
compliance = pd.read_csv(BASE_SPARK + 'compliancemetrics.csv')
noncompliant = compliance[compliance['compliancestatusv2'].isin(['LATE', 'MISSING'])]
print(f"Total non-compliant trials: {len(noncompliant)}")
print(noncompliant['compliancestatusv2'].value_counts())

Total non-compliant trials: 92757
compliancestatusv2
MISSING    68351
LATE       24406
Name: count, dtype: int64


In [8]:
# Load new Spark trials_clean
trials = pd.read_csv(BASE_SPARK + 'trialsclean.csv')
print(f"Trials loaded: {trials.shape}")
print(f"Columns: {list(trials.columns)}")

Trials loaded: (132185, 11)
Columns: ['nctid', 'orgname', 'orgclass', 'sponsorgroup', 'studytype', 'phases', 'overallstatus', 'primarycompletiondate', 'resultsfirstsubmitdate', 'primarydt', 'resultsdt']


/tmp/ipykernel_2350/3404531902.py:2: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  trials = pd.read_csv(BASE_SPARK + 'trialsclean.csv')


In [9]:
# Unzip ALL FAERS files
FAERS_DRIVE = BASE_OLD + '/FAERS/'
FAERS_LOCAL = '/content/faers_extracted/'
os.makedirs(FAERS_LOCAL, exist_ok=True)

zip_files = sorted(glob.glob(FAERS_DRIVE + '*.zip'))
print(f"Found {len(zip_files)} ZIP files\n")

for zf in zip_files:
    print(f"Unzipping: {os.path.basename(zf)}")
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(FAERS_LOCAL)

drug_files = sorted(glob.glob(FAERS_LOCAL + '**/DRUG*.txt', recursive=True))
print(f"\nTotal DRUG files found: {len(drug_files)}")

Found 36 ZIP files

Unzipping: faers_ascii_2017q1.zip
Unzipping: faers_ascii_2017q2.zip
Unzipping: faers_ascii_2017q3.zip
Unzipping: faers_ascii_2017q4.zip
Unzipping: faers_ascii_2018q1.zip
Unzipping: faers_ascii_2018q2.zip
Unzipping: faers_ascii_2018q3.zip
Unzipping: faers_ascii_2018q4.zip
Unzipping: faers_ascii_2019Q1.zip
Unzipping: faers_ascii_2019Q2.zip
Unzipping: faers_ascii_2019Q3.zip
Unzipping: faers_ascii_2019Q4.zip
Unzipping: faers_ascii_2020Q1.zip
Unzipping: faers_ascii_2020Q2.zip
Unzipping: faers_ascii_2020Q3.zip
Unzipping: faers_ascii_2020Q4.zip
Unzipping: faers_ascii_2021Q1.zip
Unzipping: faers_ascii_2021Q2.zip
Unzipping: faers_ascii_2021Q3.zip
Unzipping: faers_ascii_2021Q4.zip
Unzipping: faers_ascii_2022Q3.zip
Unzipping: faers_ascii_2022Q4.zip
Unzipping: faers_ascii_2022q1.zip
Unzipping: faers_ascii_2022q2.zip
Unzipping: faers_ascii_2023Q3.zip
Unzipping: faers_ascii_2023Q4.zip
Unzipping: faers_ascii_2023q1.zip
Unzipping: faers_ascii_2023q2.zip
Unzipping: faers_ascii_2024Q

In [11]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, countDistinct
from collections import Counter

spark = SparkSession.builder \
    .appName("TrialWatch-FAERS") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print(f"Processing {len(drug_files)} DRUG files with Spark...\n")

ae_counter = Counter()
total_rows = 0
total_ps = 0

for f in drug_files:
    try:
        df = spark.read.csv(f, header=True, sep="$", inferSchema=False)
        rows = df.count()
        total_rows += rows

        # Filter to Primary Suspect, clean, and count
        ps = df.filter(col("role_cod") == "PS")
        ps_count = ps.count()
        total_ps += ps_count

        counts = ps.withColumn("prod_ai_clean", upper(trim(col("prod_ai")))) \
            .groupBy("prod_ai_clean") \
            .agg(countDistinct("primaryid").alias("ae_count")) \
            .toPandas()

        for _, row in counts.iterrows():
            ae_counter[row['prod_ai_clean']] += row['ae_count']

        print(f"  {os.path.basename(f)}: {rows} rows ({ps_count} PS)")
    except Exception as e:
        print(f"  SKIPPED {os.path.basename(f)}: {e}")

ae_counts = pd.DataFrame(
    ae_counter.most_common(),
    columns=['active_ingredient', 'ae_count']
)

print(f"\nTotal FAERS rows processed: {total_rows}")
print(f"Total Primary Suspect rows: {total_ps}")
print(f"Unique active ingredients: {len(ae_counts)}")
print(f"\nTop 15:")
print(ae_counts.head(15).to_string())

spark.stop()
print("\nSpark session stopped!")

Processing 36 DRUG files with Spark...

  DRUG20Q1.txt: 1943532 rows (460317 PS)
  DRUG20Q2.txt: 1825414 rows (429221 PS)
  DRUG20Q3.txt: 1895153 rows (431666 PS)
  DRUG20Q4.txt: 1918927 rows (436142 PS)
  DRUG21Q1.txt: 2208416 rows (463735 PS)
  DRUG21Q2.txt: 2291903 rows (479934 PS)
  DRUG21Q3.txt: 2260570 rows (504149 PS)
  DRUG21Q4.txt: 1778675 rows (412544 PS)
  DRUG22Q1.txt: 1994171 rows (461618 PS)
  DRUG22Q2.txt: 1828103 rows (435373 PS)
  DRUG22Q3.txt: 1835461 rows (446271 PS)
  DRUG22Q4.txt: 2006967 rows (483377 PS)
  DRUG23Q1.txt: 1899503 rows (432073 PS)
  DRUG23Q2.txt: 1885096 rows (418531 PS)
  DRUG23Q3.txt: 1768391 rows (407533 PS)
  DRUG23Q4.txt: 1920732 rows (415438 PS)
  DRUG24Q1.txt: 1909327 rows (406273 PS)
  DRUG24Q2.txt: 1888937 rows (398134 PS)
  DRUG24Q3.txt: 1907293 rows (408860 PS)
  DRUG24Q4.txt: 2030938 rows (434165 PS)
  DRUG25Q1.txt: 2008162 rows (431702 PS)
  DRUG25Q2.txt: 1829056 rows (428772 PS)
  DRUG25Q3.txt: 2148451 rows (480527 PS)
  DRUG25Q4.txt: 1

In [12]:
# Fetch drug names for ALL non-compliant trials
def fetch_interventions_batch(nct_ids):
    results = {}
    ids_string = ",".join(nct_ids)
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "filter.ids": ids_string,
        "fields": "NCTId,InterventionName",
        "pageSize": 100
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            for study in data.get("studies", []):
                nct_id = study.get("protocolSection", {}).get("identificationModule", {}).get("nctId", "")
                interventions = study.get("protocolSection", {}).get("armsInterventionsModule", {}).get("interventions", [])
                names = [i.get("name", "") for i in interventions if i.get("name")]
                results[nct_id] = "; ".join(names) if names else None
    except Exception as e:
        print(f"Error: {e}")
    return results

nct_list = noncompliant['nctid'].tolist()
batches = [nct_list[i:i+100] for i in range(0, len(nct_list), 100)]
print(f"Fetching drug names for {len(nct_list)} trials in {len(batches)} batches...")

all_interventions = {}
for idx, batch in enumerate(batches):
    result = fetch_interventions_batch(batch)
    all_interventions.update(result)

    if (idx + 1) % 100 == 0:
        print(f"  Completed {idx + 1}/{len(batches)} batches ({len(all_interventions)} found)")

    time.sleep(0.3)

print(f"\nDone! Got drug names for {len(all_interventions)} out of {len(nct_list)} trials")

Fetching drug names for 92757 trials in 928 batches...
  Completed 100/928 batches (10000 found)
  Completed 200/928 batches (20000 found)
  Completed 300/928 batches (30000 found)
  Completed 400/928 batches (40000 found)
  Completed 500/928 batches (50000 found)
  Completed 600/928 batches (60000 found)
  Completed 700/928 batches (70000 found)
  Completed 800/928 batches (80000 found)
  Completed 900/928 batches (90000 found)

Done! Got drug names for 92757 out of 92757 trials


In [13]:
# Build enriched dataset
noncompliant = noncompliant.copy()
noncompliant['intervention_names'] = noncompliant['nctid'].map(all_interventions)

# Merge org info from trials_clean
noncompliant = noncompliant.merge(
    trials[['nctid', 'orgname', 'orgclass', 'phases']],
    on='nctid',
    how='left'
)

print(f"Shape: {noncompliant.shape}")
print(f"Have drug names: {noncompliant['intervention_names'].notna().sum()}")
print(f"Missing drug names: {noncompliant['intervention_names'].isna().sum()}")

# Save checkpoint
noncompliant.to_csv(BASE_OLD + '/full_noncompliant_with_drugs_spark.csv', index=False)
print("\nCheckpoint saved!")

Shape: (92757, 11)
Have drug names: 92756
Missing drug names: 1

Checkpoint saved!


In [14]:
# Drug name cleaning function
def clean_drug_names(intervention_string):
    if pd.isna(intervention_string):
        return []
    drugs = intervention_string.split(';')
    cleaned = []
    for drug in drugs:
        drug = drug.strip()
        skip_words = ['placebo', 'sham', 'control', 'no treatment', 'standard of care',
                      'usual care', 'waitlist', 'wait list', 'observation']
        if any(s in drug.lower() for s in skip_words):
            continue
        drug = re.sub(r'\([\d\.\s]*(mg|ml|mcg|g|iu|units?)[\s/\w]*\)', '', drug, flags=re.IGNORECASE)
        alt_match = re.search(r'\(([^)]+)\)', drug)
        alt_name = alt_match.group(1).strip() if alt_match else None
        drug = re.sub(r'\([^)]*\)', '', drug).strip()
        drug = re.sub(r'\b(low|high|moderate|standard)\s+dose\b', '', drug, flags=re.IGNORECASE)
        drug = re.sub(r'\b(oral|iv|injectable|topical)\s*(tablet|capsule|cream|solution)?\b', '', drug, flags=re.IGNORECASE)
        drug = drug.strip().upper()
        if drug and len(drug) > 1:
            cleaned.append(drug)
        if alt_name and len(alt_name) > 1:
            cleaned.append(alt_name.upper())
    return cleaned

# Match trials to FAERS
faers_ingredients = set(ae_counts['active_ingredient'].dropna())
matches = []

for i, row in noncompliant.iterrows():
    drug_names = clean_drug_names(row['intervention_names'])
    best_ae_count = 0
    best_match = None

    for drug in drug_names:
        if drug in faers_ingredients:
            count = ae_counts[ae_counts['active_ingredient'] == drug]['ae_count'].values[0]
            if count > best_ae_count:
                best_ae_count = count
                best_match = drug
        else:
            partial = ae_counts[ae_counts['active_ingredient'].str.contains(drug, na=False, regex=False)]
            if len(partial) > 0:
                top = partial.iloc[0]
                if top['ae_count'] > best_ae_count:
                    best_ae_count = top['ae_count']
                    best_match = top['active_ingredient']

    matches.append({
        'nctid': row['nctid'],
        'matched_ingredient': best_match,
        'ae_count': best_ae_count if best_match else 0
    })

    if (i + 1) % 10000 == 0:
        print(f"  Matched {i + 1} trials...")

matches_df = pd.DataFrame(matches)
matched = matches_df['matched_ingredient'].notna().sum()
print(f"\nMatched to FAERS: {matched} ({matched/len(matches_df)*100:.1f}%)")
print(f"No match: {len(matches_df) - matched}")

  Matched 10000 trials...
  Matched 20000 trials...
  Matched 30000 trials...
  Matched 40000 trials...
  Matched 50000 trials...
  Matched 60000 trials...
  Matched 70000 trials...
  Matched 80000 trials...
  Matched 90000 trials...

Matched to FAERS: 40409 (43.6%)
No match: 52348


In [15]:
# Add danger scores
noncompliant = noncompliant.merge(matches_df, on='nctid', how='left')

noncompliant['danger_score_raw'] = np.log1p(noncompliant['ae_count'])
max_score = noncompliant['danger_score_raw'].max()
noncompliant['danger_score'] = ((noncompliant['danger_score_raw'] / max_score) * 100).round(1)

noncompliant['danger_tier'] = pd.cut(
    noncompliant['ae_count'],
    bins=[-1, 0, 100, 1000, 10000, float('inf')],
    labels=['NO DATA', 'LOW', 'MODERATE', 'HIGH', 'CRITICAL']
)

print("Danger tier breakdown:")
print(noncompliant['danger_tier'].value_counts().to_string())

# Save checkpoint
noncompliant.to_csv(BASE_OLD + '/full_with_danger_scores_spark.csv', index=False)
print("\nCheckpoint saved!")

Danger tier breakdown:
danger_tier
NO DATA     52348
CRITICAL    19515
HIGH        11303
MODERATE     5409
LOW          4182

Checkpoint saved!


In [16]:
# Query NIH Reporter for funding
def query_nih_funding(org_name):
    url = "https://api.reporter.nih.gov/v2/projects/search"
    payload = {
        "criteria": {"org_names": [org_name]},
        "limit": 50,
        "offset": 0
    }
    try:
        resp = requests.post(url, json=payload, timeout=15)
        if resp.status_code == 200:
            data = resp.json()
            results = data.get("results", [])
            total = sum(r.get("award_amount", 0) or 0 for r in results)
            count = len(results)
            return total, count
        return 0, 0
    except:
        return 0, 0

org_counts = noncompliant['orgname'].value_counts()
orgs_to_query = org_counts[org_counts >= 2].index.tolist()
print(f"Querying NIH for {len(orgs_to_query)} organizations...")

funding_records = []
for i, org in enumerate(orgs_to_query):
    amount, count = query_nih_funding(org)
    funding_records.append({
        'orgname': org,
        'public_dollars_at_risk': amount,
        'funding_count': count
    })
    if (i + 1) % 100 == 0:
        print(f"  Completed {i+1}/{len(orgs_to_query)}")
    time.sleep(0.2)

funding_df = pd.DataFrame(funding_records)
funded = funding_df[funding_df['public_dollars_at_risk'] > 0]
print(f"\nOrgs with NIH funding: {len(funded)}")
print(f"Total public dollars at risk: ${funded['public_dollars_at_risk'].sum():,.0f}")

Querying NIH for 6618 organizations...
  Completed 100/6618
  Completed 200/6618
  Completed 300/6618
  Completed 400/6618
  Completed 500/6618
  Completed 600/6618
  Completed 700/6618
  Completed 800/6618
  Completed 900/6618
  Completed 1000/6618
  Completed 1100/6618
  Completed 1200/6618
  Completed 1300/6618
  Completed 1400/6618
  Completed 1500/6618
  Completed 1600/6618
  Completed 1700/6618
  Completed 1800/6618
  Completed 1900/6618
  Completed 2000/6618
  Completed 2100/6618
  Completed 2200/6618
  Completed 2300/6618
  Completed 2400/6618
  Completed 2500/6618
  Completed 2600/6618
  Completed 2700/6618
  Completed 2800/6618
  Completed 2900/6618
  Completed 3000/6618
  Completed 3100/6618
  Completed 3200/6618
  Completed 3300/6618
  Completed 3400/6618
  Completed 3500/6618
  Completed 3600/6618
  Completed 3700/6618
  Completed 3800/6618
  Completed 3900/6618
  Completed 4000/6618
  Completed 4100/6618
  Completed 4200/6618
  Completed 4300/6618
  Completed 4400/6618
  

In [17]:
# Final merge and save risk_enrichment.csv
noncompliant = noncompliant.merge(funding_df, on='orgname', how='left')
noncompliant['public_dollars_at_risk'] = noncompliant['public_dollars_at_risk'].fillna(0)
noncompliant['funding_count'] = noncompliant['funding_count'].fillna(0)

risk_enrichment = noncompliant[[
    'nctid', 'orgname', 'compliancestatusv2', 'matched_ingredient',
    'ae_count', 'danger_score', 'danger_tier',
    'public_dollars_at_risk', 'funding_count'
]].copy()

risk_enrichment.to_csv(BASE_OLD + '/risk_enrichment.csv', index=False)

print("FINAL risk_enrichment.csv saved!")
print(f"\nShape: {risk_enrichment.shape}")
print(f"Trials with danger scores: {(risk_enrichment['ae_count'] > 0).sum()}")
print(f"Trials with funding data: {(risk_enrichment['public_dollars_at_risk'] > 0).sum()}")
print(f"\nDanger tier breakdown:")
print(risk_enrichment['danger_tier'].value_counts().to_string())
print(f"\nTop 10 highest-risk trials:")
print(risk_enrichment.nlargest(10, ['danger_score', 'public_dollars_at_risk']).to_string())

FINAL risk_enrichment.csv saved!

Shape: (92757, 9)
Trials with danger scores: 40409
Trials with funding data: 24619

Danger tier breakdown:
danger_tier
NO DATA     52348
CRITICAL    19515
HIGH        11303
MODERATE     5409
LOW          4182

Top 10 highest-risk trials:
             nctid                                  orgname compliancestatusv2 matched_ingredient  ae_count  danger_score danger_tier  public_dollars_at_risk  funding_count
787    NCT05964465     Medical University of South Carolina            MISSING          DUPILUMAB    491306         100.0    CRITICAL              39387436.0           50.0
3705   NCT03679676                      Stanford University            MISSING          DUPILUMAB    491306         100.0    CRITICAL              32406589.0           50.0
53116  NCT06012448                   University of Michigan            MISSING          DUPILUMAB    491306         100.0    CRITICAL              26756887.0           50.0
8497   NCT04988022  Icahn School of 

In [18]:
# Add deduplication columns for accurate rollups
risk = pd.read_csv(BASE_OLD + '/risk_enrichment.csv')

risk['ae_count_deduped'] = 0
first_drug = risk.dropna(subset=['matched_ingredient']).drop_duplicates(subset=['matched_ingredient'])
risk.loc[first_drug.index, 'ae_count_deduped'] = first_drug['ae_count']

risk['funding_deduped'] = 0
first_org = risk[risk['public_dollars_at_risk'] > 0].drop_duplicates(subset=['orgname'])
risk.loc[first_org.index, 'funding_deduped'] = first_org['public_dollars_at_risk']

print(f"BEFORE (inflated):")
print(f"  Total AEs: {risk['ae_count'].sum():,.0f}")
print(f"  Total funding: ${risk['public_dollars_at_risk'].sum():,.0f}")

print(f"\nAFTER (deduplicated):")
print(f"  Total unique AEs: {risk['ae_count_deduped'].sum():,.0f}")
print(f"  Total unique funding: ${risk['funding_deduped'].sum():,.0f}")

risk.to_csv(BASE_OLD + '/risk_enrichment.csv', index=False)
print("\nFixed risk_enrichment.csv saved!")

BEFORE (inflated):
  Total AEs: 1,047,058,986
  Total funding: $419,478,103,908

AFTER (deduplicated):
  Total unique AEs: 13,988,689
  Total unique funding: $11,086,668,535

Fixed risk_enrichment.csv saved!
